# CodeSwitch-TTS — real-data scale + duration-conditioned draft (v6)

This is the Kaggle run sheet for **plan 014**. It does five things:

1. clones/pulls the GitHub repo at a pinned branch and prints the exact commit;
2. installs only missing data-path packages (Kaggle's Torch/NumPy stack is untouched);
3. streams real speech (public **FLEURS Tamil smoke** by default; gated
   **Rasa / IndicVoices-R** once `HF_TOKEN` is present);
4. encodes DAC CB0–8 shards and CTC per-character durations with
   `prepare_real_speech.py`;
5. trains a **v6 duration-conditioned draft** (optionally the residual) and
   logs config, git commit, data statistics, metrics, and artifacts to W&B.

It does **not** lower the `rt_a >= 5%` / `rd_a >= 5%` gate and does not claim
B/C paper quality. The gate diagnostic and full evaluation are opt-in.

## Before Run All

1. In Kaggle: **Settings → Accelerator → GPU T4 x2** (or P100) and
   **Internet → On**.
2. In **Add-ons → Secrets**:
   - `WANDB_API_KEY` — required for online W&B; an offline fallback is provided.
   - `HF_TOKEN` — required only for gated `ai4bharat/Rasa` or
     `ai4bharat/indicvoices_r`. Not needed for FLEURS.
   - `GITHUB_TOKEN` — only if the GitHub repo is private.
3. **Push your branch before running.** This notebook clones `REPO_REF` from
   GitHub; uncommitted local edits are invisible to Kaggle.
4. Click **Run All**. The default is a small public FLEURS Tamil smoke.
   Set `DATASET_CHOICE = "rasa_ta"` and `SMOKE = False` for the real corpus.
   Rows are not hours: 200 Rasa rows gave only **0.29 h**, so a 200-row gate
   result is a plumbing/underfitting result, not a model verdict. The real
   default is 5000 rows capped at **6 h** by `PREP_MAX_HOURS`.

## To sync code changes

Edit locally → `git push` → re-run the clone cell and the downstream cells.
The notebook prints the exact commit SHA and logs it to W&B.

## Stop rules (unchanged)

- Do not report B/C quality from this notebook alone.
- The code gate is still `rt_a >= 5%` **and** `rd_a >= 5%` from
  `diagnose_audio_bug.py`; set `DO_GATE = True` to run it.
- Do not overwrite `codeswitch_ckpt_v2`, `codeswitch_ckpt_v3`,
  `codeswitch_ckpt_v4_draft`, or the existing `teacher_cache`.
- `codeswitch_ckpt_v6` is a new output directory.


## 1. Configuration

Defaults are deliberately a **smoke**: 10 public FLEURS Tamil utterances, a
2-epoch draft, no residual, no gate, no 60-utterance evaluation. For a real
paper run set `SMOKE = False`, choose `rasa_ta`, provide `HF_TOKEN`, and
expect the draft/residual runs to use the existing wall-clock budgets.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/tripathiji1312/codeSwitch.git"
REPO_REF = "fix/013-bc-noise"
REPO_DIR = Path("/kaggle/working/codeSwitch")

SMOKE = True
DATASET_CHOICE = "fleurs_ta"  # "fleurs_ta" | "rasa_ta" | "indicvoices_ta"

SOURCES = {
    "fleurs_ta": {
        "dataset": "google/fleurs",
        "config": "ta_in",
        "split": "validation",
        "language": "ta",
        "gated": False,
        "label": "FLEURS Tamil (public smoke; not paper data)",
    },
    "rasa_ta": {
        "dataset": "ai4bharat/Rasa",
        "config": "Tamil",
        "split": "train",
        "language": "ta",
        "gated": True,
        "label": "Rasa Tamil (gated; real corpus)",
    },
    "indicvoices_ta": {
        "dataset": "ai4bharat/indicvoices_r",
        "config": "Tamil",
        "split": "train",
        "language": "ta",
        "gated": True,
        "label": "IndicVoices-R Tamil (gated; real corpus)",
    },
}

if DATASET_CHOICE not in SOURCES:
    raise ValueError(f"DATASET_CHOICE must be one of {sorted(SOURCES)}")
SOURCE = dict(SOURCES[DATASET_CHOICE])

DATA_DIR = REPO_DIR / "real_cache" / DATASET_CHOICE
CKPT_DIR = REPO_DIR / "codeswitch_ckpt_v6"
GATE_DIR = REPO_DIR / "eval_results_v6_gate"
ARG_DIR = REPO_DIR / "eval_results_v6_argmax"
ARG_AUDIO = REPO_DIR / "eval_audio_v6_argmax"

# Data prep. Rows are NOT hours: 200 Rasa rows produced only ~0.29 h. The real
# slice is bounded by PREP_MAX_HOURS; 5000 rows is only an upper row cap, and the
# source may stop earlier. A <1 h corpus is a plumbing smoke, not a gate run.
PREP_LIMIT = 10 if SMOKE else 5000
PREP_MAX_HOURS = 0.3 if SMOKE else 6.0
MIN_DURATION = 0.6
MAX_DURATION = 30.0
MIN_CHARS = 4
STRICT_DURATIONS = False  # True means "fail if any row lacks CTC durations"

# Draft / residual.  These are smoke budgets; the real command is in the markdown.
DRAFT_EPOCHS = 2 if SMOKE else 300
DRAFT_BATCH = 4 if SMOKE else 16
MIN_EPOCHS = 0 if SMOKE else 150
MAX_MINUTES = 5.0 if SMOKE else 90.0

DO_RESIDUAL = False
RESIDUAL_EPOCHS = 2 if SMOKE else 150
RESIDUAL_RAMP_EPOCHS = 1 if SMOKE else 100
MAX_MINUTES_RESIDUAL = 5.0 if SMOKE else 90.0

DO_PREP = True
DO_TRAIN = True
DO_GATE = False
DO_EVAL = False
ALLOW_EVAL_WITHOUT_GATE = False
DO_ZIP = True
INCLUDE_DATA_SHARDS_IN_ZIP = False

WANDB_PROJECT = "codeswitch-tts"
WANDB_ENTITY = None  # str or None
WANDB_GROUP = "v6-realdata"
LOG_MODEL_ARTIFACT = True
ASR_MODEL = "openai/whisper-large-v3"
SEED = 42

WANDB_URLS = []
GATE_PASS = None
train_meta = None

print("source:", SOURCE)
print("data dir:", DATA_DIR)
print("checkpoint dir:", CKPT_DIR)
print("smoke:", SMOKE, "| prep limit:", PREP_LIMIT, "| strict durations:", STRICT_DURATIONS)


## 2. Secrets and helper

Secrets are read from Kaggle Secrets first and from environment variables as a
fallback. Values are never printed. If `WANDB_API_KEY` is missing the notebook
sets W&B offline and tells you how to sync at the end.


In [ ]:
import json
import os
import shlex
import subprocess
import sys
import zipfile

def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value
    except Exception as exc:
        print(f"[secrets] {name} not in Kaggle Secrets ({type(exc).__name__})")
    return os.environ.get(name)

for _name in ("WANDB_API_KEY", "HF_TOKEN", "GITHUB_TOKEN"):
    _value = _secret(_name)
    if _value:
        os.environ[_name] = _value

print("WANDB_API_KEY:", "set (hidden)" if os.environ.get("WANDB_API_KEY") else "missing -> offline")
print("HF_TOKEN:", "set (hidden)" if os.environ.get("HF_TOKEN") else "missing")
print("GITHUB_TOKEN:", "set (hidden)" if os.environ.get("GITHUB_TOKEN") else "missing")

if SOURCE["gated"] and not os.environ.get("HF_TOKEN"):
    raise RuntimeError(
        f"{SOURCE['dataset']} is gated. Accept its terms on the Hugging Face Hub, "
        "add HF_TOKEN to Kaggle Secrets, or keep DATASET_CHOICE='fleurs_ta' for the smoke."
    )
if not os.environ.get("WANDB_API_KEY"):
    os.environ.setdefault("WANDB_MODE", "offline")

PYTHON = sys.executable

def run(cmd, cwd=None, check=True, env=None):
    cmd = [str(part) for part in cmd]
    print("$ " + " ".join(shlex.quote(part) for part in cmd))
    proc = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        env=env if env is not None else os.environ.copy(),
    )
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(cmd)}")
    return proc


## 3. Clone / pull the repo

The branch must already be pushed. A private repo needs `GITHUB_TOKEN`; the
token is passed through a temporary `GIT_ASKPASS` helper and never written into
the URL or printed.


In [ ]:
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
git_env = os.environ.copy()
askpass_path = None
if os.environ.get("GITHUB_TOKEN"):
    askpass_path = Path("/tmp/codeswitch-git-askpass.sh")
    askpass_path.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding="utf-8",
    )
    askpass_path.chmod(0o700)
    git_env["GIT_ASKPASS"] = str(askpass_path)
    git_env["GIT_TERMINAL_PROMPT"] = "0"

if (REPO_DIR / ".git").is_dir():
    print("repo exists; fetching", REPO_REF)
    fetch = run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=False, env=git_env)
    if fetch.returncode != 0:
        raise RuntimeError(
            f"Could not fetch origin/{REPO_REF}. Push the branch first, or set "
            "REPO_REF='main' after merging. See the git output above."
        )
    run(["git", "checkout", "-B", REPO_REF, "FETCH_HEAD"], cwd=REPO_DIR, env=git_env)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a git checkout; move it away or set REPO_DIR.")
    run(
        ["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)],
        cwd=REPO_DIR.parent,
        env=git_env,
    )

for required in ("prepare_real_speech.py", "train_draft_distill.py", "run_full_evaluation.py"):
    if not (REPO_DIR / required).is_file():
        raise RuntimeError(
            f"checkout is missing {required}; the branch does not contain plan 014. "
            "Push the new code and re-run this cell."
        )

sys.path.insert(0, str(REPO_DIR))
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
status = subprocess.check_output(["git", "status", "--short"], cwd=REPO_DIR, text=True).strip()
print("commit:", commit)
print(subprocess.check_output(["git", "log", "--oneline", "-3"], cwd=REPO_DIR, text=True))
print("working tree:", "clean" if not status else status)


## 4. Install only the missing data-path packages

`requirements-kaggle.txt` deliberately does **not** pin or downgrade NumPy,
pandas, pyarrow, Transformers, tokenizers, or huggingface-hub. Kaggle's Python
3.13 image ships a NumPy 2.x / pandas pair built against the same C ABI;
forcing `numpy<2` produces `ValueError: numpy.dtype size changed`.

The cell below checks imports first. If every data-path dependency is already
present, it does not run pip at all. When something is missing, it installs the
narrow set, which still leaves Torch/torchaudio and `descript-audio-codec`
alone. `descript-audio-codec` pulls `descript-audiotools` (protobuf<3.20), which
conflicts with the repo; this path uses `transformers.DacModel`.

If an older install already damaged this environment, start a **fresh Kaggle
session**. A kernel restart may keep the downgraded packages.


In [ ]:
import importlib.metadata as md
import importlib.util


def _version(dist):
    try:
        return md.version(dist)
    except Exception:
        return "MISSING"


def _has(module):
    try:
        return importlib.util.find_spec(module) is not None
    except Exception:
        return False


print("python:", sys.version.split()[0])
for dist in (
    "torch", "torchaudio", "numpy", "pandas", "pyarrow", "transformers",
    "datasets", "soundfile", "scipy", "sentencepiece", "wandb",
):
    print(f"  {dist:14s} {_version(dist)}")

if not _has("torch") or not _has("torchaudio"):
    raise RuntimeError(
        "torch/torchaudio are missing. Enable a Kaggle GPU image "
        "(Settings -> Accelerator) before running this notebook."
    )

runtime = {
    "transformers": "transformers",
    "datasets": "datasets",
    "numpy": "numpy",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "soundfile": "soundfile",
    "scipy": "scipy",
    "sentencepiece": "sentencepiece",
    "wandb": "wandb",
}
missing = sorted(dist for module, dist in runtime.items() if not _has(module))
if missing:
    print("installing missing:", ", ".join(missing))
    run(
        [
            PYTHON, "-m", "pip", "install", "-q",
            "--upgrade-strategy", "only-if-needed",
            "-r", "requirements-kaggle.txt",
        ],
        cwd=REPO_DIR,
    )
else:
    print("all data-path dependencies already present; no pip changes")

check = run(
    [
        PYTHON, "-c",
        "import numpy as np, pandas as pd, pyarrow, datasets, soundfile; "
        "pd.DataFrame({'x': np.arange(3)}); print('scientific stack imports OK')",
    ],
    cwd=REPO_DIR,
    check=False,
)
if check.returncode != 0:
    raise RuntimeError(
        "Kaggle's NumPy/pandas ABI is inconsistent. This normally means an older "
        "install downgraded NumPy under a pandas built for NumPy 2. Start a "
        "fresh Kaggle session (a kernel restart may not restore packages), then "
        "re-run this cell. Do not install numpy<2."
    )
print("dependency check complete")


In [ ]:
import torch
import transformers
import datasets
import soundfile
import pyarrow
import wandb

print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("wandb:", wandb.__version__)

if os.environ.get("WANDB_API_KEY"):
    try:
        wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
        print("W&B login: ok")
    except Exception as exc:
        print(f"W&B login failed ({type(exc).__name__}); using offline mode")
        os.environ["WANDB_MODE"] = "offline"
else:
    print("W&B: offline mode (no WANDB_API_KEY)")

if not torch.cuda.is_available():
    print("WARNING: no GPU. Data prep still works on CPU; training will be very slow.")

prep_help = subprocess.check_output(
    [PYTHON, "prepare_real_speech.py", "--help"], cwd=REPO_DIR, text=True
)
train_help = subprocess.check_output(
    [PYTHON, "train_draft_distill.py", "--help"], cwd=REPO_DIR, text=True
)
if "--durations" not in prep_help or "ctc" not in prep_help:
    raise RuntimeError("prepare_real_speech.py is stale; push/re-run the branch.")
if "--draft-duration-conditioning" not in train_help or "--require-durations" not in train_help:
    raise RuntimeError("train_draft_distill.py is stale; push/re-run the branch.")
print("feature flags: CTC durations + duration-conditioned draft present")


## 5. Real-speech → DAC shards

The pipeline writes the shared `shard_*.npz` contract: 9 DAC codebooks plus
optional `char_durations` (DAC frames, one per normalized text character),
`duration_source`, alignment score/coverage, source, speaker, and language.
Old teacher shards still load unchanged.

Default CTC model is
`Harveenchadha/vakyansh-wav2vec2-tamil-tam-250` (public, character CTC,
50 Hz). English uses `facebook/wav2vec2-large-960h-lv60-self` when an English
row is present. The DAC frame rate is `44100 / 512 = 86.1328`, not the rounded
`87` in the DAC config.

This cell logs a W&B data-prep run and the small `meta.json`; it does not
upload the shards by default.


In [ ]:
meta_path = DATA_DIR / "meta.json"

if DO_PREP:
    if SOURCE["gated"]:
        from datasets import get_dataset_config_names
        names = get_dataset_config_names(SOURCE["dataset"], token=os.environ.get("HF_TOKEN"))
        if SOURCE["config"] not in names:
            matches = [n for n in names if "tamil" in n.lower() or n.lower() in {"ta", "ta_in"}]
            if not matches:
                raise RuntimeError(
                    f"{SOURCE['dataset']} has no Tamil config; configs seen: {names[:20]}"
                )
            SOURCE["config"] = matches[0]
            print("using dataset config:", SOURCE["config"])

    prep_cmd = [
        PYTHON, "prepare_real_speech.py",
        "--out", str(DATA_DIR),
        "--hf-dataset", SOURCE["dataset"],
        "--hf-config", SOURCE["config"],
        "--hf-split", SOURCE["split"],
        "--language", SOURCE["language"],
        "--source-name", f"real:{DATASET_CHOICE}",
        "--durations", "ctc",
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--limit", str(PREP_LIMIT),
        "--max-hours", str(PREP_MAX_HOURS),
        "--min-duration", str(MIN_DURATION),
        "--max-duration", str(MAX_DURATION),
        "--min-chars", str(MIN_CHARS),
        "--log-every", "20",
    ]
    if STRICT_DURATIONS:
        prep_cmd.append("--require-durations")
    run(prep_cmd, cwd=REPO_DIR)
else:
    print("DO_PREP=False; using existing cache", DATA_DIR)

if not meta_path.is_file():
    raise RuntimeError(f"data prep wrote no meta.json: {meta_path}")

data_meta = json.loads(meta_path.read_text(encoding="utf-8"))
stats = data_meta.get("stats", {})
print("data summary")
for key in ("shards", "hours", "duration_shards", "mean_duration_coverage", "sources", "languages"):
    print(f"  {key}: {data_meta.get(key)}")
print("  stats:", {k: stats.get(k) for k in ("rows_seen", "written", "skipped_existing", "errors", "duration_ok", "duration_missing")})
data_hours = float(data_meta.get("hours", 0.0) or 0.0)
if not SMOKE and data_hours < 1.0:
    print(
        "WARNING: this real corpus is only "
        f"{data_hours:.3f} h (< 1 h). The unchanged 5% gate is dominated by data "
        "scale here; do not interpret its failure as a model-architecture verdict. "
        "Increase PREP_LIMIT / PREP_MAX_HOURS or use a larger source."
    )

shards = sorted(DATA_DIR.glob("shard_*.npz"))
if not shards:
    raise RuntimeError(f"no shards in {DATA_DIR}")
if DO_TRAIN and data_meta.get("duration_shards", 0) == 0:
    print("WARNING: no shard has CTC durations; the duration head will have no supervised target.")
if STRICT_DURATIONS and data_meta.get("duration_shards", 0) != len(shards):
    raise RuntimeError("STRICT_DURATIONS=True but some shards lack durations")

if DO_PREP:
    run_kwargs = {"entity": WANDB_ENTITY} if WANDB_ENTITY else {}
    data_run = wandb.init(
        project=WANDB_PROJECT,
        job_type="data-prep",
        name=f"data-{DATASET_CHOICE}",
        group=WANDB_GROUP,
        config={
            "dataset": SOURCE["dataset"],
            "config": SOURCE["config"],
            "split": SOURCE["split"],
            "language": SOURCE["language"],
            "limit": PREP_LIMIT,
            "max_hours": PREP_MAX_HOURS,
            "data_hours": data_hours,
            "git_commit": commit,
            "smoke": SMOKE,
        },
        reinit=True,
        **run_kwargs,
    )
    data_run.summary["git_commit"] = commit
    data_run.summary["shards"] = int(data_meta.get("shards", 0))
    data_run.summary["hours"] = float(data_meta.get("hours", 0.0))
    data_run.summary["duration_shards"] = int(data_meta.get("duration_shards", 0))
    data_run.summary["mean_duration_coverage"] = float(data_meta.get("mean_duration_coverage", 0.0))
    artifact = wandb.Artifact(f"real-speech-meta-{DATASET_CHOICE}", type="dataset")
    artifact.add_file(str(meta_path))
    data_run.log_artifact(artifact)
    if data_run.url:
        WANDB_URLS.append(data_run.url)
    print("W&B data run:", data_run.url or data_run.dir)
    data_run.finish()


## 6. Train the v6 duration-conditioned draft

This is the same `DraftTransformer` plus:

- a `DurationPredictor` head (log1p DAC frames per character),
- a duration-conditioned `FrameDecoder` projection,
- a smooth-L1 duration loss on rows that have a CTC alignment.

`--require-durations` is opt-in. With mixed data the duration loss is masked to
the aligned rows; the rest still train the CB0 head and use a uniform frame
layout. The checkpoint records `duration_shards`, `duration_shard_fraction`, and
`val_duration_loss`; `val_top1` is inference-honest (predicted durations) and
the oracle-duration pass is printed separately.

**Smoke defaults:** 2 epochs, no residual. For a paper run set `SMOKE = False`,
`DO_RESIDUAL = True`, and use the real `rasa_ta` cache.

The real floor is 150 epochs for both stages. The previous real run used a
shared floor of 100, so the residual never reached its 150-epoch budget. The
residual log now prints its learning rate; if the LR collapses while training
loss is still falling, the run is optimization-limited rather than converged.


In [ ]:
if DO_TRAIN:
    if not shards:
        raise RuntimeError("no training shards; run the data-prep cell first")
    run_kwargs = {"entity": WANDB_ENTITY} if WANDB_ENTITY else {}
    train_run = wandb.init(
        project=WANDB_PROJECT,
        job_type="draft-v6",
        name=f"draft-v6-{DATASET_CHOICE}",
        group=WANDB_GROUP,
        config={
            "dataset": SOURCE["dataset"],
            "config": SOURCE["config"],
            "git_commit": commit,
            "duration_conditioning": True,
            "duration_loss_weight": 0.1,
            "draft_epochs": DRAFT_EPOCHS,
            "draft_batch": DRAFT_BATCH,
            "min_epochs": MIN_EPOCHS,
            "do_residual": DO_RESIDUAL,
            "data_hours": data_hours,
            "smoke": SMOKE,
        },
        reinit=True,
        **run_kwargs,
    )
    train_run.summary["git_commit"] = commit
    train_run.summary["data_dir"] = str(DATA_DIR)
    train_run.summary["shards"] = len(shards)
    train_run.summary["data_hours"] = data_hours

    train_cmd = [
        PYTHON, "train_draft_distill.py",
        "--skip-teacher",
        "--teacher-cache", str(DATA_DIR),
        "--ckpt-dir", str(CKPT_DIR),
        "--device", "cuda:0" if torch.cuda.is_available() else "cpu",
        "--dtype", "fp16" if torch.cuda.is_available() else "fp32",
        "--seed", str(SEED),
        "--draft-frame-decoder-layers", "1",
        "--draft-duration-conditioning",
        "--draft-duration-loss-weight", "0.1",
        "--draft-epochs", str(DRAFT_EPOCHS),
        "--draft-batch", str(DRAFT_BATCH),
        "--min-epochs", str(MIN_EPOCHS),
        "--max-minutes", str(MAX_MINUTES),
    ]
    if STRICT_DURATIONS:
        train_cmd.append("--require-durations")
    if DO_RESIDUAL:
        train_cmd += [
            "--residual-epochs", str(RESIDUAL_EPOCHS),
            "--residual-ramp-epochs", str(RESIDUAL_RAMP_EPOCHS),
            "--residual-sched", "1.0",
            "--residual-d-model", "512",
            "--residual-layers", "6",
            "--max-minutes-residual", str(MAX_MINUTES_RESIDUAL),
        ]
    else:
        train_cmd.append("--skip-residual")

    try:
        run(train_cmd, cwd=REPO_DIR)
    except Exception:
        train_run.finish(exit_code=1)
        raise

    train_meta_path = CKPT_DIR / "meta.json"
    if not train_meta_path.is_file():
        train_run.finish(exit_code=1)
        raise RuntimeError(f"training wrote no meta.json: {train_meta_path}")
    train_meta = json.loads(train_meta_path.read_text(encoding="utf-8"))
    draft_stats = train_meta.get("draft", {})
    numeric = {k: float(v) for k, v in draft_stats.items() if isinstance(v, (int, float))}
    train_run.log(numeric)
    train_run.summary["duration_shards"] = int(train_meta.get("duration_shards", 0))
    train_run.summary["duration_shard_fraction"] = float(train_meta.get("duration_shard_fraction", 0.0))
    train_run.summary["val_top1"] = float(draft_stats.get("val_top1", 0.0))
    train_run.summary["val_top5"] = float(draft_stats.get("val_top5", 0.0))

    if LOG_MODEL_ARTIFACT:
        model_artifact = wandb.Artifact(f"codeswitch-v6-{DATASET_CHOICE}", type="model")
        model_artifact.add_file(str(CKPT_DIR / "draft_cb0.pt"))
        if (CKPT_DIR / "residual_cb1_8.pt").is_file():
            model_artifact.add_file(str(CKPT_DIR / "residual_cb1_8.pt"))
        model_artifact.add_file(str(train_meta_path))
        train_run.log_artifact(model_artifact)

    if train_run.url:
        WANDB_URLS.append(train_run.url)
    print("checkpoint:", CKPT_DIR)
    print("draft stats:", {k: draft_stats.get(k) for k in (
        "val_top1", "val_top5", "val_duration_loss", "duration_shards",
        "duration_shard_fraction", "val_top1_gt_durations", "val_top5_gt_durations",
    )})
    print("W&B train run:", train_run.url or train_run.dir)
    train_run.finish()
else:
    train_meta_path = CKPT_DIR / "meta.json"
    if train_meta_path.is_file():
        train_meta = json.loads(train_meta_path.read_text(encoding="utf-8"))
    print("DO_TRAIN=False; existing checkpoint:", CKPT_DIR)


## 7. Gate diagnostic (opt-in, unchanged 5% threshold)

`diagnose_audio_bug.py` is the existing code-level gate. It is not redefined
here:

- `rt_a >= 5.0%` — residual top-1 with teacher CB0, argmax.
- `rd_a >= 5.0%` — residual top-1 with draft CB0, argmax.

Set `DO_GATE = True` after a residual checkpoint exists. A failure is printed
as FAIL and blocks `DO_EVAL`; do not lower it.


In [ ]:
if DO_GATE:
    if not (CKPT_DIR / "residual_cb1_8.pt").is_file():
        raise RuntimeError("DO_GATE=True requires a residual checkpoint; set DO_RESIDUAL=True")
    gate_shards = min(4, len(shards))
    gate_cmd = [
        PYTHON, "diagnose_audio_bug.py",
        "--ckpt", str(CKPT_DIR),
        "--teacher-cache", str(DATA_DIR),
        "--shards", str(gate_shards),
        "--out", str(GATE_DIR),
        "--device", "cuda:0" if torch.cuda.is_available() else "cpu",
    ]
    run(gate_cmd, cwd=REPO_DIR)
    report_path = GATE_DIR / "diagnostic_report.json"
    if not report_path.is_file():
        raise RuntimeError(f"gate wrote no report: {report_path}")
    gate_report = json.loads(report_path.read_text(encoding="utf-8"))
    gate_summary = gate_report["summary"]
    GATE_PASS = bool(gate_summary["rt_a"] >= 5.0 and gate_summary["rd_a"] >= 5.0)
    print("GATE", "PASS" if GATE_PASS else "FAIL", gate_summary)

    run_kwargs = {"entity": WANDB_ENTITY} if WANDB_ENTITY else {}
    gate_run = wandb.init(
        project=WANDB_PROJECT,
        job_type="gate-diagnostic",
        name=f"gate-v6-{DATASET_CHOICE}",
        group=WANDB_GROUP,
        config={
            "git_commit": commit,
            "threshold_rt_a": 5.0,
            "threshold_rd_a": 5.0,
            "data_hours": data_hours,
        },
        reinit=True,
        **run_kwargs,
    )
    gate_run.log(gate_summary)
    gate_run.summary["gate_pass"] = GATE_PASS
    gate_run.summary["git_commit"] = commit
    gate_artifact = wandb.Artifact(f"gate-report-{DATASET_CHOICE}", type="evaluation")
    gate_artifact.add_file(str(report_path))
    gate_run.log_artifact(gate_artifact)
    if gate_run.url:
        WANDB_URLS.append(gate_run.url)
    print("W&B gate run:", gate_run.url or gate_run.dir)
    gate_run.finish()
else:
    print("DO_GATE=False; gate diagnostic skipped")


## 8. Full argmax evaluation (opt-in)

This calls the existing `run_full_evaluation.py` with real DAC/ASR provenance
and pushes to W&B. It is expensive and is not the same thing as the 5% gate.
Leave it off until the draft/residual checkpoint is worth evaluating.


In [ ]:
if DO_EVAL:
    if not DO_GATE and not ALLOW_EVAL_WITHOUT_GATE:
        raise RuntimeError(
            "DO_EVAL requires DO_GATE=True so the 5% gate runs first. "
            "Set ALLOW_EVAL_WITHOUT_GATE=True only for an explicit diagnostic run."
        )
    if DO_GATE and GATE_PASS is False:
        raise RuntimeError("Gate failed; refusing to run the 60-utterance evaluation. Do not lower the gate.")
    if not (CKPT_DIR / "residual_cb1_8.pt").is_file():
        raise RuntimeError("DO_EVAL=True requires a residual checkpoint; set DO_RESIDUAL=True")
    eval_cmd = [
        PYTHON, "run_full_evaluation.py",
        "--ckpt", str(CKPT_DIR),
        "--configs", "B,C",
        "--sample-mode", "argmax",
        "--results", str(ARG_DIR / "raw_results.jsonl"),
        "--audio-dir", str(ARG_AUDIO),
        "--figures-dir", str(ARG_DIR),
        "--asr-model", ASR_MODEL,
        "--force",
        "--wandb",
        "--wandb-project", WANDB_PROJECT,
        "--wandb-group", WANDB_GROUP,
        "--wandb-run-name", f"v6-eval-{DATASET_CHOICE}",
    ]
    run(eval_cmd, cwd=REPO_DIR)
    summary_path = ARG_DIR / "summary.json"
    if summary_path.is_file():
        print("evaluation summary:", summary_path)
        print(summary_path.read_text(encoding="utf-8")[:4000])
    else:
        print("WARNING: evaluation wrote no summary.json")
else:
    print("DO_EVAL=False; 60-utterance evaluation skipped")


## 9. Bundle outputs and final summary

The zip contains the v6 checkpoint(s), training meta, data `meta.json` /
`manifest.jsonl`, and any gate/eval reports. Raw shards are excluded by
default (`INCLUDE_DATA_SHARDS_IN_ZIP=True` to include them); they are large and
can be regenerated. W&B artifacts are the primary backup when online; the zip
is the `/kaggle/working` download.


In [ ]:
if DO_ZIP:
    bundle_root = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else REPO_DIR.parent
    bundle_path = bundle_root / f"codeswitch_v6_{DATASET_CHOICE}.zip"
    include = [CKPT_DIR, GATE_DIR, ARG_DIR]
    if INCLUDE_DATA_SHARDS_IN_ZIP:
        include.append(DATA_DIR)
    else:
        include += [DATA_DIR / "meta.json", DATA_DIR / "manifest.jsonl"]
    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for item in include:
            if not item.exists():
                continue
            if item.is_dir():
                for child in sorted(item.rglob("*")):
                    if child.is_file():
                        archive.write(child, child.relative_to(REPO_DIR))
            else:
                archive.write(item, item.relative_to(REPO_DIR))
    print("bundle:", bundle_path, f"({bundle_path.stat().st_size / 1e6:.1f} MB)")
else:
    print("DO_ZIP=False; no bundle written")


In [ ]:
print("=" * 78)
print("CodeSwitch-TTS v6 real-data run summary")
print("  git commit :", commit)
print("  source     :", SOURCE["label"])
print("  data       :", DATA_DIR)
print("  shards     :", len(shards))
print("  data hours :", data_hours)
print("  durations  :", data_meta.get("duration_shards"), "/", len(shards),
      "coverage", data_meta.get("mean_duration_coverage"))
print("  checkpoint :", CKPT_DIR if DO_TRAIN else "not trained")
if train_meta is not None:
    print("  draft top1 :", train_meta.get("draft", {}).get("val_top1"))
    print("  draft top5 :", train_meta.get("draft", {}).get("val_top5"))
    print("  duration loss:", train_meta.get("draft", {}).get("val_duration_loss"))
if DO_GATE:
    print("  gate pass  :", GATE_PASS)
print("  W&B        :")
if WANDB_URLS:
    for url in WANDB_URLS:
        print("   ", url)
else:
    print("    no online URLs recorded")
if os.environ.get("WANDB_MODE") == "offline":
    print("  offline W&B: run `wandb sync <run-dir>` from the printed run directory")
print("  next       : inspect the gate; do not claim B/C quality if rt_a/rd_a < 5%")
print("=" * 78)
